# Neuro MVP 6 — Phase 6B Demo (Colab T4 GPU)

This notebook runs the **Phase 6B** reaching experiment end-to-end on a
Colab **T4 GPU** runtime and renders an MP4 of the brain reaching to a
target using a 2-link MuJoCo arm simulated with MJX.

**Pipeline**
1. Install `mujoco`, `mujoco-mjx`, `equinox`, `imageio[ffmpeg]`.
2. Upload or clone the `Neuro_MVP_6` repo.
3. Confirm JAX sees the T4 GPU.
4. Build the reacher (`build_reacher`) — brain + MjxArmBody.
5. Motor **babbling** (30 000 cycles, OU exploration).
6. **Reaching** training (2000 episodes).
7. **Sleep cycle** (SWS replay + REM rollout).
8. Render a highlight reach episode to an MP4 video and display it inline.

> Requires **Runtime → Change runtime type → T4 GPU**.

## 1. Install dependencies

In [11]:
!pip -q install "jax[cuda12]" equinox mujoco mujoco-mjx "imageio[ffmpeg]" numpy

In [12]:
!nvidia-smi

Tue Apr 21 18:58:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             29W /   70W |   11387MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the repo
Either upload the project as a zip, clone from GitHub, or mount Drive.

In [13]:
# Option A: upload a zip of the repo through the file browser then unzip:
# !unzip -q Neuro_MVP_6.zip -d /content/
# %cd /content/Neuro_MVP_6

# Option B: clone from GitHub (adjust URL):
# !git clone https://github.com/<you>/Neuro_MVP_6.git /content/Neuro_MVP_6
# %cd /content/Neuro_MVP_6

# Option C: mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SNN

import os, sys
sys.path.insert(0, os.getcwd())
print('CWD =', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SNN
CWD = /content/drive/MyDrive/SNN


## 3. JAX GPU sanity check

In [14]:
import jax
print('JAX backend:', jax.default_backend())
print('Devices :', jax.devices())
assert 'gpu' in jax.default_backend() or jax.default_backend() == 'cuda', (
    'T4 GPU not detected — use Runtime → Change runtime type → T4 GPU.'
)

JAX backend: gpu
Devices : [CudaDevice(id=0)]


## 4. Build the reacher (brain + MjxArmBody)

In [15]:
import time
import numpy as np
import jax
import jax.numpy as jnp

from core.backend import DEFAULT, make_key
from embodiment.reacher_env import build_reacher
from embodiment.mjx_run_loop import (
    run_reach_episode, run_babbling, collect_render_frames,
)

params, state, body = build_reacher(make_key(0))
print('brain_state OK — cortex L5 =', state.cortex.rate_l5.shape)
print('body joints=', body.cfg.n_joints, 'motor_dim=', body.cfg.motor_dim)

brain_state OK — cortex L5 = (32,)
body joints= 2 motor_dim= 2


## 5. Motor babbling (30 000 cycles)

OU-noise-driven exploration of the arm workspace to train the cerebellum
forward model and the cortical state embedding.

In [ ]:
# TEST KRYTYCZNY: 2 kroków ucieleśnionych
print("Testuję 2 kroków (Mózg + MJX)...")
t_one = time.time()

bab_test = run_babbling(
    state, params, DEFAULT, body, jax.random.PRNGKey(0), 
    n_cycles=2
)

print(f"Sukces!  zajęło {time.time()-t_one:.2f}s.")

# t0 = time.time()
# bab = run_babbling(
#     state, params, DEFAULT, body, make_key(1),
#     n_cycles=30_000, ou_tau=20.0, ou_sigma=0.4,
#     target_refresh=1000,
# )
# state, body = bab.brain_state, bab.body
# print(f'babbling: {time.time()-t0:.1f}s for 30k cycles')
# tips = np.asarray(bab.tip_traj)
# print('tip bbox:', tips.min(0), tips.max(0))

Testuję 1 pełny krok (Mózg + MJX)...
Sukces! 1 krok ucieleśniony zajął 164.35s.


## 6. Reaching (2 000 episodes)

In [ ]:
N_EPISODES = 2000
MAX_STEPS  = 300
success_curve = []
mean_dist_curve = []
key = make_key(2)
t0 = time.time()
for ep in range(N_EPISODES):
    key, k_ep = jax.random.split(key)
    res = run_reach_episode(
        state, params, DEFAULT, body, k_ep,
        max_steps=MAX_STEPS, reset_body=True,
    )
    state, body = res.brain_state, res.body
    success_curve.append(bool(res.success))
    mean_dist_curve.append(float(jnp.mean(res.dists[-50:])))
    if (ep + 1) % 100 == 0:
        recent = np.mean(success_curve[-100:])
        print(f'ep {ep+1:4d}/{N_EPISODES}  success@100={recent:.2f}  '
              f'meanD={np.mean(mean_dist_curve[-100:]):.3f}  '
              f'({time.time()-t0:.0f}s)')
print('training success rate (last 200):',
      np.mean(success_curve[-200:]))

## 7. Sleep cycle (SWS + REM via `brain_cycle`)

In [ ]:
from core.brain_graph import brain_cycle
pre_success = np.mean(success_curve[-200:])
# Force several sleep cycles by invoking brain_cycle with a low-ATP
# sensory pause signal: feed zero sensory with reward=0 so the endogenous
# sleep switch (Phase 5B) can trigger.
key = make_key(7)
zero_sensory = jnp.zeros(body.sensory_size, jnp.float32)
for _ in range(300):
    key, k = jax.random.split(key)
    state = brain_cycle(
        state, params, DEFAULT,
        sensory=zero_sensory,
        reward=jnp.asarray(0.0, jnp.float32),
        done=jnp.asarray(0.0, jnp.float32),
        key=k,
    )
print('mean ATP after sleep:',
      float(jnp.mean(state.astrocyte.atp)))

In [ ]:
post_success = []
post_dists = []
key = make_key(3)
for ep in range(100):
    key, k_ep = jax.random.split(key)
    res = run_reach_episode(
        state, params, DEFAULT, body, k_ep,
        max_steps=MAX_STEPS, reset_body=True,
    )
    state, body = res.brain_state, res.body
    post_success.append(bool(res.success))
    post_dists.append(float(jnp.mean(res.dists[-50:])))
print(f'pre-sleep success:  {pre_success:.3f}')
print(f'post-sleep success: {np.mean(post_success):.3f}')
print(f'post-sleep meanD:   {np.mean(post_dists):.3f}')

## 8. Highlight reach episode → MP4 video

We capture the full trajectory of one reach, render each frame on CPU
via `mujoco.Renderer`, and save it to `reach_highlight.mp4`.

In [ ]:
highlight = run_reach_episode(
    state, params, DEFAULT, body, make_key(4242),
    max_steps=300, reset_body=True, record_qpos=True,
)
print('highlight success =', bool(highlight.success),
      ' final d =', float(highlight.dists[-1]))
print('qpos traj shape   =', highlight.qpos_traj.shape)

In [ ]:
import imageio
frames = collect_render_frames(
    body,
    highlight.qpos_traj,
    highlight.target_traj,
    width=480, height=480, camera='topdown',
)
print('rendered', len(frames), 'frames')
imageio.mimsave('reach_highlight.mp4', frames, fps=30, quality=7)
print('saved reach_highlight.mp4')

In [ ]:
import base64, pathlib
from IPython.display import HTML
mp4 = pathlib.Path('reach_highlight.mp4').read_bytes()
b64 = base64.b64encode(mp4).decode('ascii')
HTML(f'<video width=480 controls autoplay loop>'
     f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

---
**Done.**  The video above shows a biologically-grounded action brain
(cortex/thalamus/BG/cerebellum/M1/proprio) reaching to a target using
an MJX-simulated 2-link arm, with no backpropagation anywhere in the
learning pipeline.